In [1]:
import numpy as np
import math
from mpi4py import MPI
from petsc4py import PETSc

from dolfinx import mesh, fem, io 
import dolfinx.fem.petsc
import ufl
from basix.ufl import element
import ufl.constant

In [ ]:
domain_mesh = mesh.create_unit_square(
    MPI.COMM_WORLD,
    150, 
    150,
    cell_type=mesh.CellType.triangle
)


mu = 10.
k = 1.


V = fem.functionspace(
    domain_mesh, 
    ("CG", 2, (domain_mesh.geometry.dim, ))
)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

Q = fem.functionspace(
    domain_mesh, 
    ("CG", 1)
)

VQ = basix.ufl

p = ufl.TrialFunction(Q)
q = ufl.TestFunction(Q)


In [10]:
tol = 1e-8

# Boundary marker functions.
def inflow(x):
    return np.isclose(x[0], 0.0, atol=tol)

def outflow(x):
    return np.isclose(x[0], 1.0, atol=tol)

def walls(x):
    return np.logical_or(np.isclose(x[1], 0.0, atol=tol),
                         np.isclose(x[1], 1.0, atol=tol))

# Apply Dirichlet BC on the inflow boundary using the provided dofs_inflow.
dofs_inflow = fem.locate_dofs_geometrical(Q, inflow)
bc_inflow = fem.dirichletbc(PETSc.ScalarType(1.0), dofs_inflow, Q)

dofs_outflow = fem.locate_dofs_geometrical(Q, outflow)
bc_outflow = fem.dirichletbc(PETSc.ScalarType(0.0), dofs_outflow, Q)

# For velocity, we want to enforce free‐slip on the walls, i.e. only the y‐component is set to zero.
# First, locate all dofs for the velocity (subspace 0) on the wall boundaries.
dofs_walls_all = fem.locate_dofs_geometrical(V, walls)
# --- Assuming an interleaved ordering of the 2D vector dofs,
# the y‑component is stored in every second entry (i.e. indices 1, 3, 5, ...).
dofs_walls_y = dofs_walls_all[dofs_walls_all % 2 == 1]
bc_walls = fem.dirichletbc(PETSc.ScalarType(0.0), dofs_walls_y, Q)

# Gather all boundary conditions.
bcs = [bc_walls, bc_inflow, bc_outflow]

# Gather all boundary conditions.
bcs = [bc_walls, bc_inflow, bc_outflow]

In [14]:
dx = ufl.dx(domain=domain_mesh)

a = ufl.dot((mu / k) * v, u) * dx # + ufl.inner(ufl.grad(p), v) * dx + q * ufl.div(u) * dx
L = q * 0. * dx

problem = fem.petsc.LinearProblem(
    a, L, bcs=bcs,
    petsc_options={"ksp_type": "preonly",
                    "pc_type": "lu"}
)
w_sol = problem.solve()

IndexError: list index out of range

In [22]:
try:
    from petsc4py import PETSc

    import dolfinx

    if not dolfinx.has_petsc:
        print("This demo requires DOLFINx to be compiled with PETSc enabled.")
        exit(0)
except ModuleNotFoundError:
    print("This demo requires petsc4py.")
    exit(0)

from mpi4py import MPI

import numpy as np

from basix.ufl import element, mixed_element
from dolfinx import default_real_type, fem, io, mesh
from dolfinx.fem.petsc import LinearProblem
from ufl import Measure, SpatialCoordinate, TestFunctions, TrialFunctions, div, exp, inner

msh = mesh.create_unit_square(MPI.COMM_WORLD, 32, 32, mesh.CellType.quadrilateral)

k = 1
Q_el = element("BDMCF", msh.basix_cell(), k, dtype=default_real_type)
P_el = element("DG", msh.basix_cell(), k - 1, dtype=default_real_type)
V_el = mixed_element([Q_el, P_el])
V = fem.functionspace(msh, V_el)

(sigma, u) = TrialFunctions(V)
(tau, v) = TestFunctions(V)

x = SpatialCoordinate(msh)
f = 10.0 * exp(-((x[0] - 0.5) * (x[0] - 0.5) + (x[1] - 0.5) * (x[1] - 0.5)) / 0.02)

dx = Measure("dx", msh)
a = inner(sigma, tau) * dx + inner(u, div(tau)) * dx + inner(div(sigma), v) * dx
L = -inner(f, v) * dx

# Get subspace of V
V0 = V.sub(0)

fdim = msh.topology.dim - 1
facets_top = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1.0))
Q, _ = V0.collapse()
dofs_top = fem.locate_dofs_topological((V0, Q), fdim, facets_top)


def f1(x):
    values = np.zeros((2, x.shape[1]))
    values[1, :] = np.sin(5 * x[0])
    return values


f_h1 = fem.Function(Q)
f_h1.interpolate(f1)
bc_top = fem.dirichletbc(f_h1, dofs_top, V0)


facets_bottom = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0.0))
dofs_bottom = fem.locate_dofs_topological((V0, Q), fdim, facets_bottom)


def f2(x):
    values = np.zeros((2, x.shape[1]))
    values[1, :] = -np.sin(5 * x[0])
    return values


f_h2 = fem.Function(Q)
f_h2.interpolate(f2)
bc_bottom = fem.dirichletbc(f_h2, dofs_bottom, V0)


bcs = [bc_top, bc_bottom]

problem = LinearProblem(
    a,
    L,
    bcs=bcs,
    petsc_options={
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "superlu_dist",
    },
)
try:
    w_h = problem.solve()
except PETSc.Error as e:  # type: ignore
    if e.ierr == 92:
        print("The required PETSc solver/preconditioner is not available. Exiting.")
        print(e)
        exit(0)
    else:
        raise e

sigma_h, u_h = w_h.split()

with io.XDMFFile(msh.comm, "out_mixed_poisson/u.xdmf", "w") as file:
    file.write_mesh(msh)
    file.write_function(u_h)

In [23]:
print(u_h.x.array)

[inf inf inf ... inf inf inf]
